# Objetivo 3 — promessa vs. entregaTrês canais de **promessa** contra um de **entrega**:```   foto de capa        (CLIP)          ─┐   description         (texto)          ├─►  reviews  (texto)   amenities           (declaração)    ─┘```A novidade em relação à versão anterior: o lado de texto deixou de ser regex epassou a ser **embedding multilíngue** (`paraphrase-multilingual-MiniLM-L12-v2`).### Por que modelo e não regexO regex que eu tinha escrito à mão cobria ~10 padrões por aspecto, em PT/EN/ES maldistribuídos. "O chuveiro pingava a noite toda" é reclamação clara e não casava comnada. E os ~13 % de reviews em outras línguas ficavam com zero em tudo — o que não é"não mencionou", é "não sei ler".O embedding multilíngue resolve os dois: projeta ~50 línguas no **mesmo espaço**,então uma frase-protótipo em português casa com um review em inglês. Validado:```"o apartamento estava muito sujo"  ↔  "the flat was really dirty"    cos = 0,81                                   ↔  "el piso estaba muy sucio"     cos = 0,84                                   ↔  "o lugar estava impecável"     cos = 0,23```E dá **simetria metodológica**: imagem e texto passam a ser medidos pela mesmaoperação — protótipo, cosseno, margem. Antes eram instrumentos de naturezasdiferentes sendo correlacionados.**O regex fica como baseline**, rodando lado a lado. Se os dois concordarem, oachado é robusto; se discordarem, descobrimos qual estava medindo errado.

## 1. Setup

In [ ]:
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy import stats
from transformers import AutoModel, AutoTokenizer

PROC  = Path("../data/processed/Airbnb")
OUT   = Path("../outputs")
MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

img  = pd.read_csv(PROC / "image_features.csv")
amen = pd.read_csv(PROC / "amenities.csv")
txt  = pd.read_csv(PROC / "listing_text.csv", low_memory=False)
rev  = pd.read_csv(PROC / "reviews_clean.csv", low_memory=False)
rev  = rev[rev.listing_id.isin(img.id)].copy()
lst  = pd.read_csv("../data/processed/listings_aggregated.csv", low_memory=False,
                   usecols=["id", "room_type", "review_scores_accuracy",
                            "review_scores_cleanliness", "review_scores_rating", "price_num"])

print(f"imagens: {len(img)} | reviews: {len(rev):,} de {rev.listing_id.nunique()} anúncios")

In [ ]:
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(MODEL)
mod = AutoModel.from_pretrained(MODEL).to(device).eval()


def encode(texts, bs=256, max_len=128):
    '''Mean pooling mascarado + L2. Equivale ao sentence-transformers sem a dependência.'''
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i + bs], padding=True, truncation=True,
                max_length=max_len, return_tensors="pt").to(device)
        with torch.no_grad():
            h = mod(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).float()
        v = (h * m).sum(1) / m.sum(1)
        out.append((v / v.norm(dim=-1, keepdim=True)).cpu().numpy())
    return np.vstack(out)


print("device:", device)

## 2. ProtótiposMesma estrutura dos prompts do CLIP: um polo positivo e um negativo por aspecto.Escritos em português — o modelo é multilíngue, então cobrem o corpus inteiro.Dois números por aspecto, por sentença:- **relevância** = `max(cos(sent, pos), cos(sent, neg))` — a frase é *sobre* isso?- **polaridade** = `cos(sent, pos) − cos(sent, neg)` — e em que direção?

In [ ]:
PROTO = {
    "cleanliness": (["o apartamento estava impecavelmente limpo",
                     "tudo muito limpo e bem cuidado"],
                    ["o apartamento estava sujo e malcuidado",
                     "havia sujeira, mofo e cheiro ruim"]),
    "space":       (["o espaço era amplo e confortável",
                     "bastante espaço para circular"],
                    ["o lugar era apertado e minúsculo",
                     "não cabia nada, muito pequeno"]),
    "light":       (["o apartamento era claro e tinha muita luz natural",
                     "ambiente ensolarado e arejado"],
                    ["o apartamento era escuro e sem luz natural",
                     "ambiente abafado e sem janela"]),
    "decor":       (["a decoração era moderna e bonita",
                     "apartamento reformado e bem decorado"],
                    ["a decoração era velha e datada",
                     "móveis gastos e acabamento antigo"]),
    "view":        (["a vista da varanda era linda",
                     "dava para ver a cidade da janela"],
                    ["não havia vista nenhuma, só parede",
                     "janela dava para um muro"]),
    "leisure":     (["a piscina e a academia do prédio eram ótimas",
                     "área de lazer completa com churrasqueira"],
                    ["o prédio não tinha área de lazer",
                     "não havia piscina nem academia"]),
    "location":    (["a localização era excelente, perto do metrô",
                     "ótima localização, tudo a pé"],
                    ["a localização era ruim e longe de tudo",
                     "difícil de chegar, mal localizado"]),
    "safety":      (["o prédio era seguro, com portaria 24 horas",
                     "me senti seguro na região"],
                    ["a região era perigosa e insegura",
                     "não me senti seguro ali"]),
    "noise":       (["o apartamento era silencioso e tranquilo",
                     "dormi muito bem, sem barulho"],
                    ["era muito barulhento, não consegui dormir",
                     "barulho de rua a noite inteira"]),
}

ASPECTS = list(PROTO)
proto_txt, proto_span = [], {}
for a, (pos, neg) in PROTO.items():
    s = len(proto_txt)
    proto_txt += pos + neg
    proto_span[a] = (s, len(pos), len(neg))

P = encode(proto_txt)
print(f"{len(ASPECTS)} aspectos, {len(proto_txt)} protótipos -> {P.shape}")

## 3. Pontuar as sentenças dos reviews

In [ ]:
SPLIT = re.compile(r"[.!?;\n]+")

sents, owner = [], []
for lid, t in zip(rev.listing_id, rev.comments.astype(str)):
    for s in SPLIT.split(t):
        s = s.strip()
        if len(s) >= 15:                      # sentença curta não carrega aspecto
            sents.append(s); owner.append(lid)

owner = np.array(owner)
print(f"{len(sents):,} sentenças de {len(rev):,} reviews")

import time
t0 = time.time()
S = encode(sents)
print(f"embeddings: {S.shape} em {time.time() - t0:.0f}s")

In [ ]:
sim = S @ P.T          # [n_sent, n_proto]

relev, polar = {}, {}
for a, (s, npos, nneg) in proto_span.items():
    cp = sim[:, s:s + npos].mean(1)
    cn = sim[:, s + npos:s + npos + nneg].mean(1)
    relev[a] = np.maximum(cp, cn)
    polar[a] = cp - cn

REL = pd.DataFrame(relev)
print("relevância por aspecto — percentis:")
print(REL.describe(percentiles=[.5, .9, .99]).T[["mean", "50%", "90%", "99%"]].round(3))

### Limiar de relevânciaUma sentença "é sobre limpeza" quando a similaridade com os protótipos de limpezapassa de um corte. Uso o **percentil 90 de cada aspecto**, calculado no própriocorpus: isso fixa a taxa de disparo em ~10 % das sentenças por aspecto, tornando osaspectos comparáveis entre si mesmo que seus protótipos tenham escalas diferentes.

In [ ]:
THR = {a: np.quantile(REL[a], 0.90) for a in ASPECTS}

rows = defaultdict(lambda: defaultdict(float))
for a in ASPECTS:
    hit = REL[a].to_numpy() >= THR[a]
    pol = polar[a]
    for lid, h, p in zip(owner, hit, pol):
        d = rows[lid]
        if h:
            d[f"{a}_n"] += 1
            d[f"{a}_pol"] += p
            if p < 0:
                d[f"{a}_neg"] += 1

n_sent = pd.Series(owner).value_counts()

out = []
for lid, d in rows.items():
    r = {"id": lid, "n_sent": int(n_sent[lid])}
    for a in ASPECTS:
        n = d[f"{a}_n"]
        r[f"emb_{a}_share"]    = n / n_sent[lid]
        r[f"emb_{a}_pol"]      = d[f"{a}_pol"] / n if n else np.nan
        r[f"emb_{a}_negshare"] = d[f"{a}_neg"] / n_sent[lid]
    out.append(r)

EMB = pd.DataFrame(out).sort_values("id").reset_index(drop=True)
print(EMB.shape)
print(EMB[[f"emb_{a}_share" for a in ASPECTS]].mean().round(3).to_string())

### Checagem de sanidadeFui mordido hoje confiando num score zero-shot sem olhar o que ele media —`img_decor` tinha virado detector de piscina. O equivalente aqui: ler as sentençasde maior similaridade com cada protótipo. Se "limpeza" trouxer frases sobre oanfitrião ser gentil, o protótipo está errado.

In [ ]:
for a in ASPECTS:
    top = np.argsort(-REL[a].to_numpy())[:3]
    pos = np.argsort(-polar[a])[:1]
    neg = np.argsort(polar[a])[:1]
    print(f"--- {a} ---")
    for i in top:
        print(f"   rel {REL[a].iloc[i]:.2f} | {sents[i][:84]}")
    print(f"   + pol {polar[a][pos[0]]:+.2f} | {sents[pos[0]][:72]}")
    print(f"   - pol {polar[a][neg[0]]:+.2f} | {sents[neg[0]][:72]}")

## 4. Baseline regex, para comparaçãoO léxico anterior, agora **com fronteira de palavra** — sem `\b`, `super` casavadentro de *supermercado* e `bom` dentro de *bombeiro*, inflando o sentimento delocalização com puro artefato.

In [ ]:
RX = {
    "cleanliness": r"limp[oa]s?|limpez\w*|suj[oa]s?|imund\w*|clean|dirty|spotless|filthy",
    "space":       r"espa[çc]os[oa]s?|ampl[oa]s?|apertad[oa]s?|min[úu]scul[oa]s?|spacious|cramped|tiny",
    "light":       r"ilumina\w*|luz natural|escur[oa]s?|bright|sunny|natural light|dark|gloomy",
    "decor":       r"decora\w*|modern[oa]s?|reformad[oa]s?|velh[oa]s?|antig[oa]s?|stylish|outdated|shabby",
    "view":        r"vista|panor[âa]mic\w*|varanda|sacada|view|skyline|balcony|terrace",
    "leisure":     r"piscina|academia|churrasqueir\w*|sauna|pool|gym|fitness|rooftop",
    "location":    r"localiza\w*|metr[ôo]|pertinho|a p[ée]|location|walking distance|subway",
    "safety":      r"seguran[çc]\w*|portaria|porteiro|perigos\w*|assalt\w*|safe|unsafe|doorman",
    "noise":       r"barulh\w*|ru[íi]d\w*|silencios[oa]s?|tranquil[oa]s?|noise|noisy|quiet",
}
RXC = {k: re.compile(r"\b(?:" + v + r")\b", re.I) for k, v in RX.items()}

for t in ["tem supermercado perto", "o bombeiro veio", "o quarto era limpo"]:
    print(f"{t!r:28s} -> {[a for a, r in RXC.items() if r.search(t)]}")

rx_rows = defaultdict(lambda: defaultdict(float))
for lid, s in zip(owner, sents):
    d = rx_rows[lid]; d["n"] += 1
    for a, r in RXC.items():
        if r.search(s):
            d[a] += 1

RXDF = pd.DataFrame([{"id": k, **{f"rx_{a}_share": v[a] / v["n"] for a in ASPECTS}}
                     for k, v in rx_rows.items()]).sort_values("id").reset_index(drop=True)
print("\n", RXDF.shape)

## 5. Description — a promessa escrita

In [ ]:
dsc = txt[txt.id.isin(img.id)][["id", "description"]].dropna()
D = encode(dsc.description.astype(str).tolist(), max_len=256)
dsim = D @ P.T

drow = {"id": dsc.id.to_numpy()}
for a, (s, npos, nneg) in proto_span.items():
    cp = dsim[:, s:s + npos].mean(1)
    cn = dsim[:, s + npos:s + npos + nneg].mean(1)
    drow[f"desc_{a}"] = cp - cn

DESC = pd.DataFrame(drow)
print(f"{len(DESC)} descriptions pontuadas")
print(DESC[[f"desc_{a}" for a in ASPECTS]].describe().T[["mean", "std"]].round(3))

## 6. Juntar

In [ ]:
INTERIOR = {"bedroom", "living", "dining", "kitchen"}

d = (img.merge(EMB, on="id", how="inner")
        .merge(RXDF, on="id", how="left")
        .merge(DESC, on="id", how="left")
        .merge(amen, on="id", how="left")
        .merge(lst, on="id", how="left"))
d["is_interior"] = d.room_pred.isin(INTERIOR)

D_ALL = d.copy()
D_INT = d[d.is_interior].copy()
n = len(D_INT)
print(f"total {len(d)} | interiores {n}")
print(f"rho detectável (80% poder, n={n}): {np.tanh(2.80 / np.sqrt(n - 3)):.3f}")

## 7. A foto concorda com os reviews?Os 6 aspectos visuais, só em fotos de interior. Benjamini-Hochberg sobre os 12 testes.

In [ ]:
VIS = ["cleanliness", "space", "light", "decor", "view", "leisure"]

def bh(df):
    df = df.sort_values("p").reset_index(drop=True)
    raw = df.p * len(df) / (df.index + 1)
    df["p_bh"] = raw[::-1].cummin()[::-1].clip(upper=1)   # BH e step-UP
    df["sig"] = df.p_bh < 0.05
    return df

res = []
for a in VIS:
    for suf in ("share", "negshare"):
        x, y = f"img_{a}", f"emb_{a}_{suf}"
        sub = D_INT[[x, y]].dropna()
        if sub[y].std() == 0:
            continue
        r, p = stats.spearmanr(sub[x], sub[y])
        res.append({"aspecto": a, "review_var": suf, "n": len(sub), "rho": r, "p": p})

R = bh(pd.DataFrame(res))
print(R.round(4).to_string(index=False))
print(f"\nsignificativos após BH: {int(R.sig.sum())} de {len(R)}")

### Modelo vs. regex — os dois concordam?

In [ ]:
cmp = []
for a in VIS:
    s1 = D_INT[[f"img_{a}", f"emb_{a}_share"]].dropna()
    s2 = D_INT[[f"img_{a}", f"rx_{a}_share"]].dropna()
    cmp.append({"aspecto": a,
                "rho_embedding": stats.spearmanr(s1.iloc[:, 0], s1.iloc[:, 1]).statistic,
                "rho_regex":     stats.spearmanr(s2.iloc[:, 0], s2.iloc[:, 1]).statistic,
                "concordancia_emb_rx": stats.spearmanr(
                    D_INT[f"emb_{a}_share"], D_INT[f"rx_{a}_share"], nan_policy="omit").statistic})
print(pd.DataFrame(cmp).round(3).to_string(index=False))

## 8. As duas promessas concordam entre si?Pergunta que não precisa de review nenhum: o que a **foto** promete bate com o quea **description** promete? Inconsistência interna é detectável antes da entrega.

In [ ]:
pp = []
for a in VIS:
    sub = D_INT[[f"img_{a}", f"desc_{a}"]].dropna()
    r, p = stats.spearmanr(sub.iloc[:, 0], sub.iloc[:, 1])
    pp.append({"aspecto": a, "n": len(sub), "rho_foto_x_desc": r, "p": p})
print(pd.DataFrame(pp).round(4).to_string(index=False))

print("\n--- description vs reviews (promessa escrita x entrega) ---")
dr = []
for a in ASPECTS:
    sub = D_INT[[f"desc_{a}", f"emb_{a}_share"]].dropna()
    if len(sub) < 30:
        continue
    r, p = stats.spearmanr(sub.iloc[:, 0], sub.iloc[:, 1])
    dr.append({"aspecto": a, "n": len(sub), "rho": r, "p": p})
print(bh(pd.DataFrame(dr)).round(4).to_string(index=False))

## 9. Triangulação com amenitiesA âncora binária. Para cada amenity verificável: entre quem **declara**, os que**mostram na foto** recebem reviews diferentes? Aqui não há prompt no lado dadeclaração — é fato.

In [ ]:
tri = []
for k, asp in [("pool", "leisure"), ("balcony", "view"), ("cityview", "view")]:
    col, dec = f"shows_{k}", f"decl_{k}"
    if dec not in D_ALL:
        continue
    sub = D_ALL[D_ALL[dec] == 1]
    if len(sub) < 30:
        continue
    hi = sub[sub[col] >= sub[col].median()]
    lo = sub[sub[col] <  sub[col].median()]
    u, p = stats.mannwhitneyu(hi[f"emb_{asp}_share"].dropna(), lo[f"emb_{asp}_share"].dropna())
    tri.append({"amenity": k, "aspecto_review": asp, "n_declara": len(sub),
                "menciona_mostra": hi[f"emb_{asp}_share"].median(),
                "menciona_nao_mostra": lo[f"emb_{asp}_share"].median(), "p": p})
print(pd.DataFrame(tri).round(4).to_string(index=False))
print("\n(entre quem DECLARA: mostrar na capa muda o quanto os hóspedes comentam?)")

## 10. Gap promessa-entrega e validação externa

In [ ]:
def z(s):
    return (s - s.mean()) / s.std(ddof=0)

for a in VIS:
    if D_INT[f"emb_{a}_negshare"].std() > 0:
        D_INT[f"gap_{a}"] = z(D_INT[f"img_{a}"]) + z(D_INT[f"emb_{a}_negshare"])
gaps = [c for c in D_INT.columns if c.startswith("gap_")]
D_INT["gap_total"] = D_INT[gaps].mean(axis=1)

print("gap_* vs review_scores_accuracy  (esperado: NEGATIVO)\n")
gg = []
for c in gaps + ["gap_total"]:
    sub = D_INT[[c, "review_scores_accuracy"]].dropna()
    r, p = stats.spearmanr(sub[c], sub.review_scores_accuracy)
    gg.append({"var": c, "rho": r, "p": p, "n": len(sub)})
print(bh(pd.DataFrame(gg)).round(4).to_string(index=False))

## 11. Gravar

In [ ]:
cols = (["id", "room_pred", "n_sent", "is_interior"]
        + [f"img_{a}" for a in VIS]
        + [c for c in D_INT.columns if c.startswith(("emb_", "rx_", "desc_", "decl_", "shows_", "gap_"))])
D_INT[cols].to_csv(PROC / "image_review_match.csv", index=False)
R.to_csv(OUT / "match_correlations.csv", index=False)
print(f"{PROC / 'image_review_match.csv'} -> {D_INT[cols].shape}")